[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaxiRuess/DeepLearning_101/blob/main/notebooks/06_Kernels/01_Vector_Add.ipynb)

# Vector Addition — Your First GPU Kernel

This notebook introduces GPU kernel programming with [Triton](https://triton-lang.org/).

We start with the simplest possible kernel: **element-wise vector addition** (`z = x + y`). Despite being trivial mathematically, it teaches the core GPU programming concepts:

1. **Grid & Block launches** — how work is divided across GPU cores
2. **Pointer arithmetic** — GPUs don't have Python lists, just raw memory addresses
3. **Masking** — handling vectors whose size isn't a perfect multiple of the block size
4. **Benchmarking** — measuring kernel performance vs PyTorch

## Why Triton?

| | CUDA C++ | Triton |
|---|---|---|
| Language | C++ with NVIDIA extensions | Python |
| Shared memory | Manual management | Automatic |
| Thread indexing | `threadIdx.x`, `blockIdx.x` | `tl.program_id`, `tl.arange` |
| Used by | NVIDIA, FlashAttention | Meta, OpenAI, most AI labs |
| Learning curve | Steep | Moderate |

Triton lets us write kernels in Python while generating GPU code that's competitive with hand-written CUDA.

## Setup

In [ ]:
import sys, os

# In Colab, clone the repo so local imports (kernels/, src/) work
if "google.colab" in str(get_ipython()):
    if not os.path.exists("/content/DeepLearning_101"):
        !git clone --depth 1 https://github.com/MaxiRuess/DeepLearning_101.git /content/DeepLearning_101
    os.chdir("/content/DeepLearning_101/notebooks/06_Kernels")
    sys.path.insert(0, "/content/DeepLearning_101")
else:
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

In [ ]:
# Detect runtime environment
import torch

IN_COLAB = "google.colab" in str(get_ipython()) if hasattr(__builtins__, '__IPYTHON__') else False
HAS_CUDA = torch.cuda.is_available()

if IN_COLAB:
    %pip install -q triton
    print(f"Running in Colab with GPU: {torch.cuda.get_device_name(0)}")
elif HAS_CUDA:
    print(f"Running locally with GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected — will use Modal for remote GPU execution")
    print("Make sure you have Modal configured: pip install modal && modal token set")

## The Kernel — Explained

Let's look at our Triton kernel step by step. The full code lives in `kernels/vector_add.py`.

In [ ]:
from kernels.vector_add import vector_add_kernel, vector_add

In [ ]:
# Read the kernel source to understand it
# (inspect.getsource doesn't work on Triton JITFunction objects)
from pathlib import Path
print(Path("kernels/vector_add.py").read_text())

### How GPU Execution Works

```
CPU says: "Launch N programs on the GPU"

GPU Grid:
┌──────────┬──────────┬──────────┬──────────┬─────┐
│ Block 0  │ Block 1  │ Block 2  │ Block 3  │ ... │
│ 1024 els │ 1024 els │ 1024 els │ 1024 els │     │
└──────────┴──────────┴──────────┴──────────┴─────┘

Each block processes BLOCK_SIZE elements independently.
All blocks run in parallel across GPU cores.
```

Key concepts:
- **`tl.program_id(0)`** — which block am I? (like `blockIdx.x` in CUDA)
- **`tl.arange(0, BLOCK_SIZE)`** — generate indices within this block
- **mask** — the last block might go past the array end, so we mask those out
- **`tl.load` / `tl.store`** — read from / write to GPU memory

## Run on GPU

Triton requires an NVIDIA GPU. This notebook supports two execution modes:
- **Colab / Local CUDA** — runs directly on the available GPU
- **Modal** — runs on a remote T4 GPU (for Mac / no-GPU machines)

In [ ]:
import time

def benchmark_vector_add():
    """Run correctness test + benchmark. Works on any CUDA device."""
    import triton
    import triton.language as tl

    @triton.jit
    def _vector_add_kernel(
        x_ptr, y_ptr, output_ptr, n_elements,
        BLOCK_SIZE: tl.constexpr,
    ):
        pid = tl.program_id(axis=0)
        block_start = pid * BLOCK_SIZE
        offsets = block_start + tl.arange(0, BLOCK_SIZE)
        mask = offsets < n_elements
        x = tl.load(x_ptr + offsets, mask=mask)
        y = tl.load(y_ptr + offsets, mask=mask)
        output = x + y
        tl.store(output_ptr + offsets, output, mask=mask)

    def _vector_add(x, y):
        output = torch.empty_like(x)
        n = output.numel()
        BLOCK_SIZE = 1024
        grid = (triton.cdiv(n, BLOCK_SIZE),)
        _vector_add_kernel[grid](x, y, output, n, BLOCK_SIZE=BLOCK_SIZE)
        return output

    # --- Correctness test ---
    torch.manual_seed(0)
    size = 100_000
    x = torch.rand(size, device="cuda")
    y = torch.rand(size, device="cuda")

    output_triton = _vector_add(x, y)
    output_torch = x + y

    max_diff = (output_triton - output_torch).abs().max().item()
    match = torch.allclose(output_triton, output_torch)
    print(f"Max difference: {max_diff}")
    print(f"Results match: {match}")

    # --- Benchmark ---
    sizes = [2**i for i in range(12, 25)]
    triton_times = []
    torch_times = []

    for s in sizes:
        x = torch.rand(s, device="cuda")
        y = torch.rand(s, device="cuda")

        for _ in range(10):
            _vector_add(x, y)
            x + y
        torch.cuda.synchronize()

        start = time.perf_counter()
        for _ in range(100):
            _vector_add(x, y)
        torch.cuda.synchronize()
        triton_times.append((time.perf_counter() - start) / 100)

        start = time.perf_counter()
        for _ in range(100):
            x + y
        torch.cuda.synchronize()
        torch_times.append((time.perf_counter() - start) / 100)

    return {
        "match": match,
        "max_diff": max_diff,
        "sizes": sizes,
        "triton_us": [t * 1e6 for t in triton_times],
        "torch_us": [t * 1e6 for t in torch_times],
    }

In [ ]:
if HAS_CUDA:
    # --- Direct GPU execution (Colab or local CUDA) ---
    results = benchmark_vector_add()
else:
    # --- Modal remote execution (no local GPU) ---
    # @triton.jit kernels can't be serialized by Modal, so all code is inline.
    import modal

    app = modal.App("triton-vector-add")
    image = modal.Image.debian_slim(python_version="3.12").pip_install("torch", "triton")

    @app.function(image=image, gpu="T4")
    def run_remote():
        import torch, triton, triton.language as tl, time

        @triton.jit
        def _vector_add_kernel(x_ptr, y_ptr, output_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
            pid = tl.program_id(axis=0)
            offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
            mask = offsets < n_elements
            x = tl.load(x_ptr + offsets, mask=mask)
            y = tl.load(y_ptr + offsets, mask=mask)
            tl.store(output_ptr + offsets, x + y, mask=mask)

        def _vector_add(x, y):
            output = torch.empty_like(x)
            n = output.numel()
            BLOCK_SIZE = 1024
            _vector_add_kernel[(triton.cdiv(n, BLOCK_SIZE),)](x, y, output, n, BLOCK_SIZE=BLOCK_SIZE)
            return output

        torch.manual_seed(0)
        size = 100_000
        x = torch.rand(size, device="cuda")
        y = torch.rand(size, device="cuda")
        output_triton = _vector_add(x, y)
        output_torch = x + y
        max_diff = (output_triton - output_torch).abs().max().item()
        match = torch.allclose(output_triton, output_torch)
        print(f"Max difference: {max_diff}, Results match: {match}")

        sizes = [2**i for i in range(12, 25)]
        triton_times, torch_times = [], []
        for s in sizes:
            x = torch.rand(s, device="cuda")
            y = torch.rand(s, device="cuda")
            for _ in range(10):
                _vector_add(x, y); x + y
            torch.cuda.synchronize()
            start = time.perf_counter()
            for _ in range(100): _vector_add(x, y)
            torch.cuda.synchronize()
            triton_times.append((time.perf_counter() - start) / 100)
            start = time.perf_counter()
            for _ in range(100): x + y
            torch.cuda.synchronize()
            torch_times.append((time.perf_counter() - start) / 100)

        return {
            "match": match, "max_diff": max_diff, "sizes": sizes,
            "triton_us": [t * 1e6 for t in triton_times],
            "torch_us": [t * 1e6 for t in torch_times],
        }

    with app.run():
        results = run_remote.remote()

In [ ]:
print(f"Correctness: {'PASS' if results['match'] else 'FAIL'}")
print(f"Max difference: {results['max_diff']:.2e}")

## Benchmark Results

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(10, 5))

ax.plot(results["sizes"], results["triton_us"], "o-", label="Triton", linewidth=2)
ax.plot(results["sizes"], results["torch_us"], "s--", label="PyTorch", linewidth=2)
ax.set_xscale("log", base=2)
ax.set_xlabel("Vector size")
ax.set_ylabel("Time (microseconds)")
ax.set_title("Vector Add: Triton vs PyTorch")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## What to Notice

1. **Triton matches PyTorch exactly** — the max difference should be 0.0 (both are doing the same floating-point add)
2. **For small vectors, PyTorch is faster** — kernel launch overhead dominates when there's little work
3. **For large vectors, performance converges** — both saturate GPU memory bandwidth
4. **Vector add is memory-bound** — the GPU spends more time loading/storing data than computing. This is why custom kernels really shine on *compute-bound* operations (like fused attention, where you avoid multiple memory round-trips)

## Resources

- [Triton Tutorials](https://triton-lang.org/main/getting-started/tutorials/index.html)
- [GPU MODE lectures](https://github.com/gpu-mode/lectures) — Community GPU programming course
- [PMPP Book](https://www.elsevier.com/books/programming-massively-parallel-processors/hwu/978-0-323-91231-0) — The classic GPU programming textbook